# 🎬 Netflix Data Analytics with AI

---

## Section 1 — Project Title

**Netflix Data Analytics with AI**  
Exploring, Visualizing, and Building a Simple ML Model on the Netflix Movies and TV Shows Dataset

---

## Section 2 — Student Information

| Field | Details |
|---|---|
| **Student Name** | Shreyashee Ghosh |
| **Program** | IBM SkillsBuild Data Analytics with AI Academic Internship |
| **Project Title** | Netflix Data Analytics with AI |
| **Dataset** | Netflix Movies and TV Shows (netflix_titles.csv) |
| **Tools Used** | Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn |

---

## Section 3 — Project Objective

The goal of this project is to perform a complete beginner-friendly Data Analytics + AI study on the **Netflix Movies and TV Shows** dataset.

**Specific objectives:**
- Load and understand the structure of the Netflix dataset.
- Clean the data by handling missing values and inconsistencies.
- Perform Exploratory Data Analysis (EDA) to uncover patterns and trends.
- Create clear, well-labelled visualizations using Matplotlib and Seaborn.
- Build a simple Machine Learning model using Scikit-learn to predict whether a title is a **Movie** or a **TV Show**.
- Evaluate the model using standard classification metrics.
- Summarize key findings and suggest future directions.

---

## Section 4 — Dataset Description

The dataset used in this project is the **Netflix Movies and TV Shows** dataset, saved as `netflix_titles.csv` in the project folder.

| Column | Description |
|---|---|
| `show_id` | Unique identifier for each title |
| `type` | Movie or TV Show |
| `title` | Name of the title |
| `director` | Director(s) of the title |
| `cast` | Lead cast members |
| `country` | Country of production |
| `date_added` | Date added to Netflix |
| `release_year` | Original release year |
| `rating` | Content rating (e.g. TV-MA, PG-13) |
| `duration` | Duration in minutes (Movies) or number of seasons (TV Shows) |
| `listed_in` | Genre categories |
| `description` | Short description of the title |

**Total records:** 8,807 titles  
**Total columns:** 12


---

## Section 5 — Importing Libraries

We start by importing all the Python libraries we need for this project.

- **pandas** — for loading and manipulating data
- **numpy** — for numerical operations
- **matplotlib** — for creating basic visualizations
- **seaborn** — for enhanced statistical visualizations
- **scikit-learn** — for building and evaluating the Machine Learning model


In [ ]:
# Standard data manipulation libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Display settings
pd.set_option('display.max_columns', 15)
pd.set_option('display.max_colwidth', 50)
%matplotlib inline

# Set a consistent plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

print("All libraries imported successfully!")

---

## Section 6 — Loading the Dataset

We load the Netflix dataset from the CSV file into a pandas DataFrame.
A DataFrame is like a table with rows and columns — similar to an Excel spreadsheet.


In [ ]:
# Load the dataset
df = pd.read_csv('netflix_titles.csv')

print(f"Dataset loaded successfully!")
print(f"Number of rows    : {df.shape[0]}")
print(f"Number of columns : {df.shape[1]}")

In [ ]:
# Display the first 5 rows of the dataset
print("First 5 rows of the dataset:")
df.head()

---

## Section 7 — Understanding the Dataset

Before cleaning and analyzing the data, it is important to understand its structure:
- How many rows and columns are there?
- What are the column names?
- What data types do the columns contain?
- Are there any missing values?
- Are there any duplicate records?


In [ ]:
# Shape of the dataset
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

In [ ]:
# Column names
print("Column names:")
print(list(df.columns))

In [ ]:
# Data types of each column
print("Data types:")
print(df.dtypes)

In [ ]:
# Statistical summary for numerical columns
print("Statistical summary:")
df.describe()

In [ ]:
# Missing values analysis
print("Missing values per column:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

**Observations from Dataset Understanding:**

- The dataset has **8,807 titles** and **12 columns**.
- The column `director` has the most missing values — **2,634 (29.9%)** — because many TV shows do not have a single credited director.
- `cast` is missing for **825 rows (9.4%)** and `country` for **831 rows (9.4%)**.
- `date_added` has only **10 missing values**, `rating` has **4**, and `duration` has **3**.
- There are **no duplicate rows** in the dataset.


---

## Section 8 — Data Cleaning and Preprocessing

Data cleaning is the process of fixing or removing incorrect, incomplete, or irrelevant data.

**Steps performed:**
1. Strip any extra whitespace from string columns.
2. Fix the three rows where the `rating` column accidentally contains duration values (`66 min`, `74 min`, `84 min`) — these are data entry errors; we will set those ratings to `NaN`.
3. Fill missing values in `director`, `cast`, and `country` with the label `'Unknown'` since these are informational columns and we do not want to drop 30% of our data.
4. Fill the very few missing values in `rating` with `'Unknown'`.
5. Drop the 3 rows where `duration` is missing — these rows also have the rating data-entry problem and are not usable.
6. Parse `date_added` to a proper datetime type and extract `year_added` and `month_added` columns.
7. Extract numeric duration: minutes for Movies, number of seasons for TV Shows.


In [ ]:
# Step 1: Strip whitespace from object columns
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

print("Whitespace stripped from all string columns.")

In [ ]:
# Step 2: Fix data-entry errors in 'rating' column
# Three rows have duration values stored in the rating column
bad_ratings = ['66 min', '74 min', '84 min']
print("Rows with bad rating values:")
print(df[df['rating'].isin(bad_ratings)][['title', 'rating', 'duration']])

# Set these erroneous rating values to NaN
df.loc[df['rating'].isin(bad_ratings), 'rating'] = np.nan
print("\nBad rating values replaced with NaN.")

In [ ]:
# Step 3: Drop rows where 'duration' is missing (only 3 rows)
df.dropna(subset=['duration'], inplace=True)
print(f"Rows after dropping missing duration: {len(df)}")

In [ ]:
# Step 4: Fill missing values in text/categorical columns
df['director'] = df['director'].fillna('Unknown')
df['cast']     = df['cast'].fillna('Unknown')
df['country']  = df['country'].fillna('Unknown')
df['rating']   = df['rating'].fillna('Unknown')

print("Missing values filled:")
print(df.isnull().sum())

In [ ]:
# Step 5: Parse date_added to datetime and extract year and month
df['date_added'] = pd.to_datetime(df['date_added'], format='%B %d, %Y', errors='coerce')
df['year_added']  = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

print("Sample of parsed date_added:")
print(df[['date_added', 'year_added', 'month_added']].head())

In [ ]:
# Step 6: Extract numeric duration
# For Movies  -> extract the number from '90 min'  -> 90
# For TV Shows -> extract the number from '2 Seasons' -> 2

df['duration_numeric'] = df['duration'].str.extract(r'(\d+)').astype(float)

print("Movies - duration_numeric sample:")
print(df[df['type'] == 'Movie'][['title', 'duration', 'duration_numeric']].head())

print("\nTV Shows - duration_numeric sample:")
print(df[df['type'] == 'TV Show'][['title', 'duration', 'duration_numeric']].head())

In [ ]:
# Final check after cleaning
print(f"Final dataset shape after cleaning: {df.shape}")
print(f"Missing values remaining:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

**Summary of data cleaning decisions:**

| Column | Issue | Treatment |
|---|---|---|
| `director` | 2,634 missing | Filled with `'Unknown'` |
| `cast` | 825 missing | Filled with `'Unknown'` |
| `country` | 831 missing | Filled with `'Unknown'` |
| `rating` | 4 missing + 3 data-entry errors | Errors set to NaN, all NaN filled with `'Unknown'` |
| `duration` | 3 missing | Rows dropped (unusable) |
| `date_added` | 10 missing, stored as string | Parsed to datetime; missing become NaT (handled gracefully) |

We chose **`'Unknown'`** for text columns instead of dropping rows because dropping 30% of the dataset (director column) would significantly distort all analyses.


---

## Section 9 & 10 — Exploratory Data Analysis and Data Visualizations

EDA is the process of visually and statistically exploring the data to find patterns, relationships, and anomalies before building any model.

We will cover:
1. Distribution of Movies vs TV Shows
2. Netflix content by release year
3. Titles added to Netflix over time
4. Top countries producing content
5. Most common genres
6. Content ratings distribution
7. Movie duration distribution
8. TV Show seasons distribution


### 9.1 — Distribution of Movies vs TV Shows


In [ ]:
type_counts = df['type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
colors = ['#e50914', '#564d4d']
axes[0].bar(type_counts.index, type_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Number of Movies vs TV Shows on Netflix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Content Type', fontsize=12)
axes[0].set_ylabel('Number of Titles', fontsize=12)
for i, v in enumerate(type_counts.values):
    axes[0].text(i, v + 40, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(
    type_counts.values,
    labels=type_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Proportion of Movies vs TV Shows', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_01_type_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Movies  : {type_counts['Movie']}  ({type_counts['Movie']/len(df)*100:.1f}%)")
print(f"TV Shows: {type_counts['TV Show']} ({type_counts['TV Show']/len(df)*100:.1f}%)")

**📊 Chart Explanation:**
The Netflix library is dominated by **Movies**, which make up about **69.7%** of all titles. TV Shows account for the remaining **30.3%**. This shows that Netflix has historically added more movies than shows, though TV Shows are a significant portion of its content.


### 9.2 — Netflix Content by Release Year


In [ ]:
# Focus on years from 2000 onwards for a cleaner chart
release_year_data = df[df['release_year'] >= 2000].groupby(['release_year', 'type']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(13, 5))
release_year_data.plot(kind='bar', ax=ax, color=['#e50914', '#564d4d'], edgecolor='white', linewidth=0.5)

ax.set_title('Netflix Titles by Release Year (2000–2021)', fontsize=14, fontweight='bold')
ax.set_xlabel('Release Year', fontsize=12)
ax.set_ylabel('Number of Titles', fontsize=12)
ax.legend(title='Content Type', fontsize=11)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('chart_02_release_year.png', dpi=120, bbox_inches='tight')
plt.show()

**📊 Chart Explanation:**
The number of titles on Netflix has grown dramatically for content released from **2014 onwards**. The peak for Movies is around **2017–2019**, reflecting both Netflix's own original movie production boom and its aggressive licensing of recent films during those years. The slight drop in 2021 is likely because many 2021 releases were still being added at the time the dataset was collected.


### 9.3 — Titles Added to Netflix Over Time


In [ ]:
titles_by_year = df.dropna(subset=['year_added']).groupby(['year_added', 'type']).size().unstack(fill_value=0)
titles_by_year = titles_by_year[titles_by_year.index >= 2015]  # Focus on meaningful range

fig, ax = plt.subplots(figsize=(12, 5))
titles_by_year['Movie'].plot(ax=ax, marker='o', color='#e50914', linewidth=2.5, label='Movie')
titles_by_year['TV Show'].plot(ax=ax, marker='s', color='#564d4d', linewidth=2.5, label='TV Show')

ax.set_title('Number of Titles Added to Netflix Per Year (2015–2021)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year Added to Netflix', fontsize=12)
ax.set_ylabel('Number of Titles Added', fontsize=12)
ax.legend(title='Content Type', fontsize=11)
ax.set_xticks(titles_by_year.index)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('chart_03_titles_added_over_time.png', dpi=120, bbox_inches='tight')
plt.show()

print("Titles added per year:")
print(titles_by_year)

**📊 Chart Explanation:**
Netflix significantly increased the number of titles it added each year from **2015 to 2019**. The number peaked in **2019** with over 2,000 titles added that year. There was a slight decline in 2020 and 2021, which may be partly explained by COVID-19 impacting production schedules and the dataset's collection cutoff date.


### 9.4 — Top 15 Countries Producing Netflix Content


In [ ]:
# Use the first listed country for each title
df['primary_country'] = df['country'].str.split(',').str[0].str.strip()

top_countries = (
    df[df['primary_country'] != 'Unknown']['primary_country']
    .value_counts()
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='Reds_r', ax=ax)

ax.set_title('Top 15 Countries by Number of Netflix Titles', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Titles', fontsize=12)
ax.set_ylabel('Country', fontsize=12)

for i, v in enumerate(top_countries.values):
    ax.text(v + 10, i, str(v), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('chart_04_top_countries.png', dpi=120, bbox_inches='tight')
plt.show()

**📊 Chart Explanation:**
The **United States** dominates Netflix content with over **2,800 titles**, followed by **India** (~970) and the **United Kingdom** (~419). This reflects both Netflix's US origins and its significant investment in Indian and British content. The presence of South Korea, Japan, Spain, and France highlights Netflix's global content strategy.


### 9.5 — Most Common Genres / Categories


In [ ]:
# Expand multi-genre entries (each title can belong to multiple genres)
all_genres = df['listed_in'].str.split(', ').explode().str.strip()
top_genres = all_genres.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, palette='Blues_r', ax=ax)

ax.set_title('Top 15 Most Common Genres on Netflix', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Titles', fontsize=12)
ax.set_ylabel('Genre', fontsize=12)

for i, v in enumerate(top_genres.values):
    ax.text(v + 10, i, str(v), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('chart_05_top_genres.png', dpi=120, bbox_inches='tight')
plt.show()

**📊 Chart Explanation:**
**International Movies** is the most common genre tag (~2,752 titles), followed by **Dramas** (~2,427) and **Comedies** (~1,674). The high count of "International Movies" reflects Netflix's global library strategy. **International TV Shows** and **Documentaries** also feature prominently, showing the platform's diverse content mix.


### 9.6 — Content Ratings Distribution


In [ ]:
rating_counts = (
    df[df['rating'] != 'Unknown']['rating']
    .value_counts()
    .reset_index()
)
rating_counts.columns = ['rating', 'count']

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=rating_counts, x='rating', y='count', palette='viridis', ax=ax)

ax.set_title('Distribution of Content Ratings on Netflix', fontsize=14, fontweight='bold')
ax.set_xlabel('Content Rating', fontsize=12)
ax.set_ylabel('Number of Titles', fontsize=12)
ax.tick_params(axis='x', rotation=45)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('chart_06_ratings.png', dpi=120, bbox_inches='tight')
plt.show()

**📊 Chart Explanation:**
**TV-MA** (Mature Audience) is the most common rating with **3,207 titles**, showing that Netflix caters heavily to adult viewers. **TV-14** follows with **2,160 titles**. Family-friendly ratings like **PG**, **TV-Y**, and **TV-Y7** are present but much smaller, suggesting Netflix's primary audience is adults and teenagers.


### 9.7 — Movie Duration Distribution


In [ ]:
movie_durations = df[df['type'] == 'Movie']['duration_numeric'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(movie_durations, bins=40, color='#e50914', edgecolor='white', linewidth=0.5)
axes[0].axvline(movie_durations.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {movie_durations.mean():.0f} min')
axes[0].axvline(movie_durations.median(), color='navy', linestyle='--', linewidth=1.5, label=f'Median: {movie_durations.median():.0f} min')
axes[0].set_title('Distribution of Movie Durations', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Duration (minutes)', fontsize=12)
axes[0].set_ylabel('Number of Movies', fontsize=12)
axes[0].legend(fontsize=11)

# Box plot
axes[1].boxplot(movie_durations, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#e50914', color='black'),
                medianprops=dict(color='white', linewidth=2))
axes[1].set_title('Box Plot of Movie Durations', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Duration (minutes)', fontsize=12)
axes[1].set_xticks([])

plt.tight_layout()
plt.savefig('chart_07_movie_duration.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Movie duration statistics:")
print(f"  Mean   : {movie_durations.mean():.1f} minutes")
print(f"  Median : {movie_durations.median():.0f} minutes")
print(f"  Min    : {movie_durations.min():.0f} minutes")
print(f"  Max    : {movie_durations.max():.0f} minutes")
print(f"  Std Dev: {movie_durations.std():.1f} minutes")

**📊 Chart Explanation:**
The distribution of movie durations on Netflix is roughly **bell-shaped**, centred around **98–100 minutes**. Most movies fall in the **75–125 minute** range. The box plot reveals some outliers on both ends — very short films (under 20 minutes) and very long films (over 200 minutes). The average movie on Netflix is approximately **99.6 minutes** long.


### 9.8 — TV Show Seasons Distribution


In [ ]:
show_seasons = df[df['type'] == 'TV Show']['duration_numeric'].dropna()
season_counts = show_seasons.value_counts().sort_index().head(10)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(season_counts.index.astype(int), season_counts.values, color='#564d4d', edgecolor='white', linewidth=0.8)

ax.set_title('TV Show Distribution by Number of Seasons', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Seasons', fontsize=12)
ax.set_ylabel('Number of TV Shows', fontsize=12)
ax.set_xticks(range(1, 11))

for i, v in zip(season_counts.index.astype(int), season_counts.values):
    ax.text(i, v + 15, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('chart_08_tv_seasons.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"TV Show season statistics:")
print(f"  Total TV Shows : {len(show_seasons)}")
print(f"  1-season shows : {int(season_counts[1])} ({season_counts[1]/len(show_seasons)*100:.1f}%)")
print(f"  Mean seasons   : {show_seasons.mean():.2f}")

**📊 Chart Explanation:**
An overwhelming majority — about **67%** — of TV shows on Netflix have only **1 season**. This reflects Netflix's strategy of adding many newer or limited-run international series. Shows with 2–3 seasons are the next most common. Very few shows have more than 5 seasons, suggesting Netflix focuses on fresh content rather than long-running series.


### 9.9 — Heatmap: Content Added by Month and Year


In [ ]:
month_year = (
    df.dropna(subset=['year_added', 'month_added'])
    .groupby(['year_added', 'month_added'])
    .size()
    .unstack(fill_value=0)
)
month_year = month_year[month_year.index >= 2016]
month_year.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    month_year,
    annot=True,
    fmt='d',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Number of Titles Added'}
)
ax.set_title('Number of Titles Added to Netflix by Month and Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Year', fontsize=12)

plt.tight_layout()
plt.savefig('chart_09_heatmap_month_year.png', dpi=120, bbox_inches='tight')
plt.show()

**📊 Chart Explanation:**
This heatmap reveals seasonal patterns in Netflix's content additions. **July, October, and December** tend to have higher additions — likely timed around summer breaks, Halloween, and the holiday season. The years **2018 and 2019** show the darkest cells, confirming they were Netflix's peak content-addition years.


### 9.10 — Content Ratings: Movies vs TV Shows (Side-by-Side)


In [ ]:
# Filter to well-known ratings
known_ratings = ['TV-MA', 'TV-14', 'TV-PG', 'R', 'PG-13', 'TV-Y7', 'TV-Y', 'PG', 'TV-G', 'NR', 'G']
rating_type_df = df[df['rating'].isin(known_ratings)]

fig, ax = plt.subplots(figsize=(13, 5))
sns.countplot(
    data=rating_type_df,
    x='rating',
    hue='type',
    order=known_ratings,
    palette=['#e50914', '#564d4d'],
    ax=ax
)

ax.set_title('Content Ratings: Movies vs TV Shows', fontsize=14, fontweight='bold')
ax.set_xlabel('Content Rating', fontsize=12)
ax.set_ylabel('Number of Titles', fontsize=12)
ax.legend(title='Content Type', fontsize=11)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('chart_10_ratings_by_type.png', dpi=120, bbox_inches='tight')
plt.show()

**📊 Chart Explanation:**
Both Movies and TV Shows are heavily concentrated in the **TV-MA** and **TV-14** ratings. Movies additionally have notable counts in **R** and **PG-13**, which are cinema-specific ratings. TV Shows are more evenly spread across the TV-specific rating scale (TV-Y to TV-MA). This side-by-side view clearly shows different rating profiles for Movies vs TV Shows.


---

## Section 11 — Key Data Analysis Findings

Based on all the EDA performed above, here are the most important findings:

| # | Finding |
|---|---|
| 1 | The Netflix dataset contains **8,807 titles** — **6,131 Movies** (69.7%) and **2,676 TV Shows** (30.3%). |
| 2 | **United States** is the largest content producer with ~2,818 titles, followed by **India** (~972) and the **UK** (~419). |
| 3 | **TV-MA** is the most common content rating (3,207 titles), confirming Netflix targets primarily adult audiences. |
| 4 | The most common genre is **International Movies** (~2,752), followed by **Dramas** (~2,427) and **Comedies** (~1,674). |
| 5 | Netflix's content additions peaked in **2019** (over 2,000 titles added that year). |
| 6 | Average movie duration is approximately **99.6 minutes**, with most movies falling between 75–125 minutes. |
| 7 | About **67% of TV Shows** have only **1 season**, suggesting Netflix favours limited-run or newer international series. |
| 8 | Content additions show **seasonal patterns**, with peaks in July, October, and December. |
| 9 | Netflix's content release year spans **1925 to 2021**, but the majority of titles were released after **2015**. |
| 10 | `director` has the highest missingness (29.9%), mainly because many TV shows lack a single credited director. |


---

## Section 12 — Machine Learning / AI Component

### What are we building?

We will build a **Classification Model** that tries to predict whether a Netflix title is a **Movie** or a **TV Show** based on its features — without using information that directly reveals the answer (such as the `duration` column which contains "min" for movies and "Seasons" for shows).

### Why this task?

Classification is one of the most common Machine Learning tasks. This is a great beginner exercise because:
- The target variable (`type`: Movie or TV Show) is clear and binary.
- We can use `release_year`, `rating`, and `year_added` as features.
- It teaches us the full ML pipeline: preprocessing → training → evaluation.

### Which algorithm?

We will use a **Decision Tree Classifier** — a simple, interpretable algorithm that makes decisions using a series of yes/no questions on the features.

### Important Rule — No Data Leakage

We will **NOT** use the `duration` column as a feature, because it directly encodes whether the title is a Movie ("min") or TV Show ("Seasons"). Using it would give artificially perfect results that don't reflect real learning.

**Features used:** `release_year`, `rating` (encoded), `year_added`  
**Target:** `type` (Movie = 0, TV Show = 1)


In [ ]:
# ── Step 1: Prepare the ML dataset ──────────────────────────────────────────

# Select features and target
ml_df = df[['type', 'release_year', 'rating', 'year_added']].copy()

# Drop rows with missing year_added (only a few rows)
ml_df.dropna(subset=['year_added'], inplace=True)

# Convert year_added to integer
ml_df['year_added'] = ml_df['year_added'].astype(int)

# Encode the 'rating' column: convert text ratings to numbers
le_rating = LabelEncoder()
ml_df['rating_encoded'] = le_rating.fit_transform(ml_df['rating'])

# Encode the target: Movie = 0, TV Show = 1
le_type = LabelEncoder()
ml_df['type_encoded'] = le_type.fit_transform(ml_df['type'])

print("Label encoding mapping for 'type':")
for label, code in zip(le_type.classes_, range(len(le_type.classes_))):
    print(f"  {label} -> {code}")

print(f"\nML dataset shape: {ml_df.shape}")
ml_df[['type', 'type_encoded', 'release_year', 'rating', 'rating_encoded', 'year_added']].head()

In [ ]:
# ── Step 2: Define features (X) and target (y) ──────────────────────────────

X = ml_df[['release_year', 'rating_encoded', 'year_added']]
y = ml_df['type_encoded']

print(f"Features (X) shape : {X.shape}")
print(f"Target (y) shape   : {y.shape}")
print(f"\nClass distribution:")
print(y.value_counts().rename(index={0: 'Movie', 1: 'TV Show'}))

In [ ]:
# ── Step 3: Split into Training and Testing sets ─────────────────────────────
# 80% of data for training, 20% for testing
# random_state=42 ensures reproducibility

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size : {X_train.shape[0]} samples")
print(f"Testing set size  : {X_test.shape[0]} samples")

In [ ]:
# ── Step 4: Train the Decision Tree Classifier ──────────────────────────────
# max_depth=5 limits tree complexity and helps prevent overfitting

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

print("Decision Tree model trained successfully!")

In [ ]:
# ── Step 5: Make Predictions ─────────────────────────────────────────────────

y_pred = dt_model.predict(X_test)

print("Predictions generated on the test set.")
print(f"First 10 actual    : {list(y_test[:10])}")
print(f"First 10 predicted : {list(y_pred[:10])}")

---

## Section 13 — Model Evaluation

We evaluate our model using the following metrics:

- **Accuracy** — What percentage of all predictions are correct?
- **Precision** — Of the titles predicted as TV Show, how many actually are TV Shows?
- **Recall** — Of all actual TV Shows, how many did the model correctly identify?
- **F1-Score** — The harmonic mean of Precision and Recall — a balanced measure.
- **Confusion Matrix** — A table showing correct vs incorrect predictions for each class.


In [ ]:
# ── Compute evaluation metrics ───────────────────────────────────────────────

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')

print("=" * 42)
print("         MODEL EVALUATION RESULTS")
print("=" * 42)
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 42)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=le_type.classes_))

In [ ]:
# ── Confusion Matrix Visualization ──────────────────────────────────────────

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Reds',
    xticklabels=le_type.classes_,
    yticklabels=le_type.classes_,
    linewidths=0.5,
    ax=ax
)

ax.set_title('Confusion Matrix — Decision Tree Classifier', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('Actual Label', fontsize=12)

plt.tight_layout()
plt.savefig('chart_11_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nHow to read the confusion matrix:")
print(f"  Correctly predicted Movies   : {cm[0][0]}")
print(f"  Movies predicted as TV Show  : {cm[0][1]}")
print(f"  TV Shows predicted as Movie  : {cm[1][0]}")
print(f"  Correctly predicted TV Shows : {cm[1][1]}")

In [ ]:
# ── Feature Importances ──────────────────────────────────────────────────────

feature_names = ['release_year', 'rating_encoded', 'year_added']
importances = dt_model.feature_importances_

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(feature_names, importances, color=['#e50914', '#564d4d', '#999999'])

ax.set_title('Feature Importances — Decision Tree', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)

for bar, imp in zip(bars, importances):
    ax.text(imp + 0.005, bar.get_y() + bar.get_height()/2,
            f'{imp:.4f}', va='center', fontsize=11)

plt.tight_layout()
plt.savefig('chart_12_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    print(f"  {name:<20}: {imp:.4f}")

**📊 Model Evaluation Explanation (Beginner-Friendly):**

The Decision Tree Classifier was trained on 3 features: `release_year`, `rating_encoded`, and `year_added`.

- The model achieves a **moderate accuracy** on the test set. This is expected because we deliberately excluded the `duration` column (which would trivially solve the problem) to build a genuinely learning model.
- The **feature importance chart** shows which features the Decision Tree relied on most when making its splits.
- The **confusion matrix** shows where the model makes mistakes — most errors occur in predicting TV Shows, which is expected since Movies outnumber TV Shows ~2.3:1 (class imbalance).
- This model is a good **educational demonstration** of the ML pipeline — it is not intended for production use.


---

## Section 14 — Important Insights

Combining all the analysis done in this project, here are the most important insights from the Netflix dataset:

### 🎬 Content Mix
- Netflix has approximately **2.3x more Movies than TV Shows**. However, TV Show production has been growing steadily, especially after 2017.

### 🌍 Global Reach
- While the **US dominates** with ~32% of all titles, **India, UK, Japan, and South Korea** together contribute a significant portion, confirming Netflix's global content expansion strategy.

### 🎭 Genre Trends
- **International Movies and Dramas** are by far the most common genres. The high count of "International" tagged content suggests Netflix actively curates content for its global subscriber base.

### 📅 Growth Pattern
- Netflix's content library grew explosively between **2016 and 2019**, tripling in size. The slight slowdown post-2019 coincides with global pandemic effects on production.

### ⏱️ Duration Patterns
- The average Netflix movie runs **~100 minutes** — very close to the standard Hollywood runtime. The vast majority of TV Shows (67%) have exactly **1 season**, reflecting Netflix's commitment to international limited series.

### 🔞 Audience Targeting
- **Over 60% of content** is rated TV-MA or TV-14, showing Netflix's strong focus on mature adult content. This aligns with Netflix's positioning as a premium adult entertainment service.

### 🤖 Machine Learning
- A Decision Tree Classifier can learn to distinguish Movies from TV Shows using only `release_year`, `rating`, and `year_added` — without being explicitly told the content type. This shows that Movies and TV Shows do have **distinct statistical profiles** in terms of the year they were released and their content ratings.


---

## Section 15 — Conclusion

In this project, we performed a complete **Data Analytics + AI** study on the Netflix Movies and TV Shows dataset for the **IBM SkillsBuild Data Analytics with AI Academic Internship**.

**What we accomplished:**

1. ✅ Loaded and explored a real-world dataset of **8,807 Netflix titles**.
2. ✅ Cleaned the data by handling missing values, fixing data-entry errors, and engineering new features (`year_added`, `month_added`, `duration_numeric`).
3. ✅ Performed thorough **Exploratory Data Analysis (EDA)** across 10 different dimensions of the data.
4. ✅ Created **12 visualizations** using Matplotlib and Seaborn with proper titles, axis labels, and explanations.
5. ✅ Identified **10 key patterns and trends** from the data.
6. ✅ Built a **Decision Tree Classifier** using Scikit-learn to predict content type (Movie vs TV Show).
7. ✅ Evaluated the model using Accuracy, Precision, Recall, F1-Score, and a Confusion Matrix.
8. ✅ Summarized findings in plain language suitable for a beginner-level data analytics project.

**Key Takeaway:**  
Netflix's content library is large, diverse, and globally-oriented — dominated by Movies for adult audiences, with a growing emphasis on international limited-run TV series. The Machine Learning component demonstrated that even simple models can learn meaningful patterns from structured data.


---

## Section 16 — Future Scope

This project can be extended in many interesting directions:

| # | Future Direction | Skill Level |
|---|---|---|
| 1 | **Sentiment Analysis on Descriptions** — Use NLP to classify titles by tone (dark, funny, romantic) based on their text descriptions. | Intermediate |
| 2 | **Multi-class Genre Prediction** — Predict the primary genre of a title using Random Forest or Gradient Boosting. | Intermediate |
| 3 | **Content Recommendation System** — Build a simple content-based recommender that suggests similar titles based on genre, country, and rating. | Intermediate |
| 4 | **Interactive Dashboard** — Build a Streamlit or Dash dashboard to make these visualizations interactive and filterable. | Intermediate |
| 5 | **Time Series Forecasting** — Predict how many titles Netflix will add in future years based on the `date_added` trend. | Advanced |
| 6 | **Deep Learning for Description Classification** — Use a BERT-based text model to classify Netflix descriptions into genres. | Advanced |
| 7 | **Network Analysis of Cast/Director** — Map co-appearances of directors and actors across titles as a social network graph. | Advanced |
| 8 | **Comparison with Competitor Datasets** — Combine with Amazon Prime or Disney+ datasets to compare content strategies. | Intermediate |

---

**Project completed by:** Shreyashee Ghosh  
**Program:** IBM SkillsBuild Data Analytics with AI Academic Internship  

---
